# Generating a Network's Weights with a Hypernetwork

Companion notebook for the blog post. Run the cells top to bottom.

In [ ]:
import polyweave
print(polyweave.__version__)

In [ ]:
import torch
from polyweave.students import make_cnn_student
from polyweave.prototypes import feature_class_stats
from polyweave.hypernets import FCMapTeacher

NUM_CLASSES, FEATURE_DIM = 10, 256

# A CNN student: frozen trunk + a generatable fc head.
student = make_cnn_student("A", feature_dim=FEATURE_DIM, num_classes=NUM_CLASSES, in_ch=3)

# The teacher that writes the fc head's {weight, bias}. sigma_pi=False = the plain
# additive teacher (a small conv net over the prototype).
teacher = FCMapTeacher(NUM_CLASSES, FEATURE_DIM, proto_channels=4, width=64, sigma_pi=False)

In [ ]:
def build_prototype(student, batch):
    x, y = batch
    feats = student.extract_features(x)            # [B, FEATURE_DIM]
    return feature_class_stats(feats, y, NUM_CLASSES)

# The forward callback ties a generated head into the student's forward pass.
def forward(student, batch, gen):
    x, y = batch
    return student(x, generated_fc=gen), y

In [ ]:
from polyweave.evaluation import (
    generate_averaged, evaluate_accuracy,
    class_centroids, centroids_to_fc, random_like,
)

gen = generate_averaged(teacher, student, support_batches, build_prototype)
acc = evaluate_accuracy(student, eval_batches, gen, forward)

# Two reference points, computed from the same support set:
centroids = class_centroids(support_feats, support_labels, NUM_CLASSES)
ncc  = evaluate_accuracy(student, eval_batches, centroids_to_fc(centroids), forward)  # strong
rand = evaluate_accuracy(student, eval_batches, random_like(gen), forward)            # floor

In [ ]:
from polyweave.evaluation import recovery_curve

curve = recovery_curve(
    model,                                   # student with the generated head installed
    init=lambda m: [m.fc.weight, m.fc.bias], # parameters to fine-tune
    sample_batch=next_support_batch,
    forward=forward,
    eval_fn=lambda m: evaluate_accuracy(m, eval_batches, None, forward),
    steps=300, lr=1e-3, eval_every=20,
)                                            # -> [(step, accuracy), ...]

In [ ]:
import torch
from polyweave.evaluation import ensemble_accuracy, ensemble_gain, pairwise_disagreement

members = []
for support in support_draws:                       # a few different support sets
    gen = generate_averaged(teacher, student, support, build_prototype)
    logits = torch.cat([forward(student, b, gen)[0] for b in eval_batches])
    members.append(logits.softmax(dim=-1))
probs = torch.stack(members)                         # [M, N, C]

print("ensemble acc :", ensemble_accuracy(probs, labels))
print("ensemble gain:", ensemble_gain(probs, labels))          # ensemble − mean member
print("diversity    :", pairwise_disagreement(probs))          # how differently members err